## MMR(Maximum Marginal Relvance)

- Reduce Redudency in document objects relevent t the query
- for diversification of returned results

In [1]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv,find_dotenv

load_dotenv(find_dotenv())



True

In [2]:
docs = [
    # Cluster A: near-duplicates (multiprocessing)
    Document(page_content="Use Python's multiprocessing module to speed up slow code by running tasks on multiple CPU cores.", metadata={"topic": "multiprocessing", "id": "A1"}),
    Document(page_content="To make Python code faster, multiprocessing lets you spread slow work across several CPU cores.", metadata={"topic": "multiprocessing", "id": "A2"}),
    Document(page_content="Slow Python scripts can be sped up with a multiprocessing Pool that runs functions in parallel on many cores.", metadata={"topic": "multiprocessing", "id": "A3"}),
    Document(page_content="Parallelize slow Python code using multiprocessing so each CPU core handles part of the workload.", metadata={"topic": "multiprocessing", "id": "A4"}),

    # Cluster B: vectorization
    Document(page_content="Replace slow Python for-loops with NumPy vectorized operations to speed up numerical code dramatically.", metadata={"topic": "numpy", "id": "B1"}),

    # Cluster C: profiling
    Document(page_content="Before optimizing, profile your Python code with cProfile to find which functions are actually slow.", metadata={"topic": "profiling", "id": "C1"}),

    # Cluster D: caching
    Document(page_content="Use functools.lru_cache to memoize expensive function calls and avoid recomputing the same results.", metadata={"topic": "caching", "id": "D1"}),

    # Cluster E: weakly related
    Document(page_content="Python was created by Guido van Rossum and first released in 1991.", metadata={"topic": "history", "id": "E1"}),
    Document(page_content="Cython compiles Python-like code to C, which can make slow numeric code run much faster.", metadata={"topic": "cython", "id": "D2"}),
]


In [3]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [4]:
vector_store=Chroma.from_documents(
    documents=docs,
    persist_directory="chroma_db",
    collection_name="mmr_retreival",
    embedding=embeddings
)

In [ ]:
retriever=vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":4,"lambda_mult":1}  
) 
#lamda
#  1.0 = pure relevance (same as similarity search)
# 0.5 = balanced
# 0.0 = maximum diversity (can pull in irrelevant docs like the Python history one)
# )

In [7]:
query="How can we speed up slow python code?"

results=retriever.invoke(query)
results

[Document(id='a33a274f-a004-4fdd-bcb1-4f328903f61b', metadata={'topic': 'multiprocessing', 'id': 'A2'}, page_content='To make Python code faster, multiprocessing lets you spread slow work across several CPU cores.'),
 Document(id='161e63e5-3793-4f98-96b4-132aa8410107', metadata={'id': 'C1', 'topic': 'profiling'}, page_content='Before optimizing, profile your Python code with cProfile to find which functions are actually slow.'),
 Document(id='2cbc28e6-70bd-4254-ba67-868991a86b81', metadata={'topic': 'multiprocessing', 'id': 'A3'}, page_content='Slow Python scripts can be sped up with a multiprocessing Pool that runs functions in parallel on many cores.'),
 Document(id='6f071805-0e97-4fc9-afbf-dff2aae88463', metadata={'topic': 'multiprocessing', 'id': 'A4'}, page_content='Parallelize slow Python code using multiprocessing so each CPU core handles part of the workload.')]

In [8]:
retriever=vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":4,"lambda_mult":0}  
) 
#lamda
#  1.0 = pure relevance (same as similarity search)
# 0.5 = balanced
# 0.0 = maximum diversity (can pull in irrelevant docs like the Python history one)
# )

In [9]:
query="How can we speed up slow python code?"

results=retriever.invoke(query)
results

[Document(id='a33a274f-a004-4fdd-bcb1-4f328903f61b', metadata={'topic': 'multiprocessing', 'id': 'A2'}, page_content='To make Python code faster, multiprocessing lets you spread slow work across several CPU cores.'),
 Document(id='59b5b606-3251-4b86-a8de-77d1f8315c34', metadata={'id': 'B1', 'topic': 'numpy'}, page_content='Replace slow Python for-loops with NumPy vectorized operations to speed up numerical code dramatically.'),
 Document(id='c32d4099-e520-431a-97ed-9b15aa8b4e83', metadata={'topic': 'caching', 'id': 'D1'}, page_content='Use functools.lru_cache to memoize expensive function calls and avoid recomputing the same results.'),
 Document(id='036988c7-a54e-4555-a8b2-6ef5482d1ee1', metadata={'id': 'E1', 'topic': 'history'}, page_content='Python was created by Guido van Rossum and first released in 1991.')]

In [11]:
retriever=vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":4,"lambda_mult":0.5}  
) 
#lamda
#  1.0 = pure relevance (same as similarity search)
# 0.5 = balanced
# 0.0 = maximum diversity (can pull in irrelevant docs like the Python history one)
# )

In [12]:
query="How can we speed up slow python code?"

results=retriever.invoke(query)
results

[Document(id='a33a274f-a004-4fdd-bcb1-4f328903f61b', metadata={'id': 'A2', 'topic': 'multiprocessing'}, page_content='To make Python code faster, multiprocessing lets you spread slow work across several CPU cores.'),
 Document(id='161e63e5-3793-4f98-96b4-132aa8410107', metadata={'id': 'C1', 'topic': 'profiling'}, page_content='Before optimizing, profile your Python code with cProfile to find which functions are actually slow.'),
 Document(id='59b5b606-3251-4b86-a8de-77d1f8315c34', metadata={'topic': 'numpy', 'id': 'B1'}, page_content='Replace slow Python for-loops with NumPy vectorized operations to speed up numerical code dramatically.'),
 Document(id='bc02bf94-9dad-4171-9f31-b1ea40ec41bd', metadata={'id': 'D2', 'topic': 'cython'}, page_content='Cython compiles Python-like code to C, which can make slow numeric code run much faster.')]